In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression ,ElasticNet
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import mean_squared_error
from sklearn.impute import SimpleImputer
from sklearn.naive_bayes import GaussianNB
from sklearn.preprocessing import OrdinalEncoder # Importing OrdinalEncoder from sklearn.preprocessing
from sklearn.svm import SVC
import scipy.stats as stats
from sklearn.metrics import accuracy_score

In [ ]:
test = pd.read_csv('/content/test.csv')
train = pd.read_csv('/content/train.csv')
ex_test= pd.read_csv('/content/training_extra.csv')
submission= pd.read_csv('/content/sample_submission.csv')


In [ ]:
train.head()

In [ ]:
train.info()

In [ ]:
train.isna().sum()

In [ ]:
train.dropna(inplace=True)
train.shape

In [ ]:
test.isna().sum()

In [ ]:
submission.head()

In [ ]:
train.drop('id', axis=1, inplace=True)
test.drop('id', axis=1, inplace=True)

train.shape, test.shape

In [ ]:
for col in train:
    if train[col].dtype == 'object':
        print(col,train[col].unique() )

In [ ]:
# Convert 'Brand' column to string type if it contains numerical values
train['Style'] = train['Style'].astype(str)

# Use plt.hist() to create histogram after conversion
plt.hist(train['Style'])
plt.xticks(rotation=90)  # Rotate x-axis labels for better readability if needed
plt.show()

In [ ]:

color_counts = train['Color'].value_counts()

# إنشاء مخطط الأعمدة
plt.figure(figsize=(10, 5))  # ضبط حجم الشكل لتحسين الوضوح
plt.bar(color_counts.index, color_counts.values, color='skyblue')  # رسم المخطط مع لون مميز


plt.xticks(rotation=90)
plt.xlabel("Color")
plt.ylabel("Count")
plt.title("Distribution of Colors in Train Data")

# عرض المخطط
plt.show()

In [ ]:
sns.displot(train['Price'], kde=True)

In [ ]:
for col in test:
    if test[col].dtype == 'object':
        test[col] = test[col].fillna('not listed')
    if test[col].dtype == 'int' or test[col].dtype == 'float':
        test[col] = test[col].fillna(-1)

test.isna().sum().sum()

In [ ]:
enc = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)

for col in train:
    if train[col].dtype == 'object':
        train[col] = enc.fit_transform(train[col].values.reshape(-1,1))
        test[col] = enc.transform(test[col].values.reshape(-1,1))

In [ ]:
train.info()

In [ ]:
train


In [ ]:
test.info()

In [ ]:
y = train.pop('Price')
X = train
X_test = test

In [ ]:
X_train, X_val, y_train, y_val = train_test_split( X, y, test_size=0.1, shuffle=True, random_state=42)
X_train.shape, y_train.shape, X_val.shape, y_val.shape, X_test.shape

In [ ]:
imputer = SimpleImputer(strategy='mean')

# Fit the imputer on the training data and transform both training and validation data
X_train = imputer.fit_transform(X_train)
X_val = imputer.transform(X_val)

# Now you can train your model
model = ElasticNet(random_state=42).fit(X_train, y_train)
model.score(X_train, y_train)

In [ ]:
y_pred = model.predict(X_val)
y_pred

In [ ]:
mse = mean_squared_error(y_val, y_pred)
rmse = np.sqrt(mse)
rmse

In [ ]:
df = pd.DataFrame({'y_val':y_val, 'y_pred':y_pred})
df

In [ ]:
stats.probplot(y_pred, dist="norm", plot=plt)
plt.show()

In [ ]:
fig, ax = plt.subplots()
ax.scatter(y_val, y_pred, edgecolors=(0, 0, 0))
ax.plot([y.min(), y.max()], [y.min(), y.max()], 'k--', lw=4)
ax.set_xlabel('Measured')
ax.set_ylabel('Predicted')
plt.show()

In [ ]:
pred = model.predict(X_test)
pred

In [ ]:

pred = model.predict(X_test)


expected_rows = 200000
actual_rows = len(pred)


if actual_rows < expected_rows:
    missing_values = expected_rows - actual_rows
    pred = np.append(pred, [-1] * missing_values)


pred = np.nan_to_num(pred, nan=-1)


submission = pd.read_csv('/content/sample_submission.csv')


if submission.shape[0] != 200000:
    missing_rows = 200000 - submission.shape[0]
    extra_rows = pd.DataFrame({'Price': [-1] * missing_rows})
    submission = pd.concat([submission, extra_rows], ignore_index=True)


submission['Price'] = pred[:200000]


submission['Price'].fillna(-1, inplace=True)


submission.to_csv('submission.csv', index=False)


submission_check = pd.read_csv('submission.csv')


print("Final Submission Shape:", submission_check.shape)
print("Total Null Values:", submission_check.isna().sum().sum())  # يجب أن يكون 0
